<img src='./img/cng_thrivegeo_banner.png' class="center" width=95%>

# 🚀 Discover Zarr

## Introduction

In the previous notebooks, you worked with single images, split into tiles, that you can read piece by piece. But Earth observation data often has more than two dimensions. A weather dataset adds **time** and **height**, an ocean model adds **depth**, and a single satellite product can hold dozens of variables at different resolutions. Keeping each time step or variable in a separate file quickly becomes hard to manage.

[Zarr](https://zarr.dev/) is a cloud-native format designed for large **N-dimensional arrays**. Instead of one big file, a Zarr store is a **folder of many small objects**:

* 📋 **Metadata**: small text files describe each array's shape, data type and chunk size, so you can explore a dataset before reading any data
* 🧊 **Chunks**: each array is cut into blocks along every dimension (e.g. one hour × the whole globe), and each block is stored as its own compressed object
* 🗂️ **Groups**: arrays can be organised in a hierarchy, like folders, e.g. Sentinel-2 reflectance at 10 m, 20 m and 60 m inside one product

If this sounds familiar, it is: the metadata plays the role of the COG header, and chunks are tiles in more dimensions. The idea is the same: **read the description first, then fetch only the pieces you need.**

How a store is cut into chunks decides what it costs to read. So throughout, you keep an eye on one number: **how many chunks you have to read** to answer a question.

Here, you will work with two datasets:

* 🌡️ **ERA5**: 6-Hourly global reanalysis data from the European Centre for Medium-Range Weather Forecasts (ECMWF). You open it from the cloud, save a small piece on your own disk, and find out why a question about **Snowbird** can cost one chunk or many more.
* 🛰️ **Sentinel-2** from the **EOPF Sentinel Zarr Samples Service**, which you met in *Discover STAC*. You find a scene through STAC and turn it into a vegetation map of the **Douro valley** in Portugal.

**Prerequisites:** basic Python, and *Discover STAC* and *Discover COG*.

### By the end of this tutorial, you will be able to...

* 💡 **Explain** how a Zarr store is organised: metadata files, chunk files and groups
* ☁️ **Open** a Zarr store from the cloud without loading data, and read only a piece of it
* 💾 **Write** a piece of a dataset as your own Zarr store
* 🧊 **Judge** from the chunk shape whether a question is cheap or expensive to answer
* 🔌 **Navigate** the groups of a store found through STAC, and combine arrays from them

### Notebook Outline

* [Example 1️⃣: Read and write a Zarr store](#example1)
* [Example 2️⃣: About Chunking](#example2)
* [Exercise 1️⃣: Explore a Sentinel-2 Zarr product](#example3)
* [Takeaways](#example4)

<hr>

**Import required libraries**

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import xarray as xr
from pystac_client import Client
from rasterio.warp import transform_bounds
from shapely.geometry import box, shape

<hr>

<a id="example1"></a>

## Example 1️⃣: Read and write a Zarr store

**ERA5** is a reconstruction of the global weather, hour by hour, since 1940. The dataset is available as Analysis-Ready Cloud-Optimised (ARCO) as one large Zarr store, called **ARCO-ERA5**: about 2 petabytes, on a 0.25° grid. Here, you will use a smaller public copy prepared by the WeatherBench 2 project: the same data on a coarser grid (1.5°, which gives 240 × 121 points), every 6 hours, from 1959 to early 2023. In this example, you:

1. ☁️ **Open** the store
2. 🧊 **Look** at one variable and its chunks
3. 🗺️ **Read** one map
4. 💾 **Write** a small piece to your own disk
5. 🔍 **Look inside** the folder you wrote



### ☁️ Step 1: Open the store

Opening a Zarr store works just like opening a local file. You only give `xarray` the address of the store. `storage_options={"token": "anon"}` tells it to connect anonymously, because the bucket is public.

In [ ]:
%%time
era5_url = "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr"

era5 = xr.open_zarr(era5_url, storage_options={"token": "anon"})
era5

The summary says the dataset is **2 TB**, yet the cell took only a couple of seconds. `xarray` has read the metadata and nothing else.

The summary of a Dataset has four parts. You will see them in every Zarr dataset you open:


| Part | What it shows | In ERA5 |
|---|---|---|
| **Dimensions** | the axes of the data, and how long each one is | `time`, `longitude`, `latitude`, `level` |
| **Coordinates** | the values along each axis | dates every 6 hours from 1959 to early 2023 |
| **Data variables** | the measurements themselves, one array each | 62 variables, e.g. `2m_temperature` |
| **Attributes** | extra information about the dataset | e.g. the source or the units |


### 🧊 Step 2: Look at one variable and its chunks

Select the 2m air temperature by its variable name `2m_temperature`, and ask three things: how long is each dimension, how large is the whole variable, and how large is one chunk?

In [ ]:
t2m = era5["2m_temperature"]
t2m

Look at the grey table in the summary. It has two columns:

* **Array**: the whole variable, all 64 years, about 10 GiB. This is how much you *would* download if you loaded everything.
* **Chunk**: one single piece of it, just under 1 MB. This is the smallest amount `xarray` can download at once.

You can also read the chunk shape straight from the metadata:

In [ ]:
t2m.encoding["chunks"]

The dimensions are in the order `(time, longitude, latitude)`. So each chunk holds **8 time steps (2 days) of the whole globe**, and each chunk is stored as one object in the bucket.

### 🗺️ Step 3: Read one time-step

How warm was the world on 1 July 2022 at noon? One time step sits inside one chunk, so this question costs **1 chunk**. `.load()` tells `xarray` to download it now:

In [ ]:
%%time
world = t2m.sel(time="2022-07-01T12:00").load()

The values are in Kelvin, so subtract 273.15 to get degrees Celsius.

In [ ]:
(world - 273.15).plot(
    x="longitude", y="latitude",
    cmap="RdBu_r", center=0,
    figsize=(9, 4.5),
    cbar_kwargs={"label": "2 m temperature (°C)"},
)
plt.title("Temperature on 1 July 2022, 12:00 UTC")
plt.xlabel("Longitude (°E)")
plt.ylabel("Latitude (°N)")
plt.show()

::: {.callout-tip}
## Reflection and answers

Reading one time-step cost you 1 chunk. What about if you extract the temperature at Snowbird for every time step in 2022:

- ➡️ 1. How many time steps does 2022 have?<br>
    ✏️ : **1,460 time steps**: 365 days × 4 time steps per day.
- ➡️ 2. How many chunks does `xarray` have to read to get the temperature time-series?<br>
    ✏️ : About **183 chunks**: 1,460 ÷ 8 = 182.5. Each of them holds the whole globe, so you would download about 170 MB to get 1,460 numbers.

The provider optimised the chunks for **spatial analysis**, not for time series.

<center>

| Question | Chunks to read |
|---|---|
| 🗺️ World map, 1 July 2022 | 1 |
| 📈 Snowbird, all of 2022 | 183 |

</center>

:::

### 💾 Step 4: Write a Zarr Store to your disk

So far, you have only read. Now select the whole year 2022 and save it as your own Zarr store. With `.sel()`, you can simply write the year as text:

In [ ]:
t2m_2022 = t2m.sel(time="2022")
t2m_2022

1,460 time steps sit in **183 chunks**. `to_zarr()` writes them to your disk, and **this is the moment the data is downloaded**.

Three details in the cell below:

* `.chunk({"time": 8})` sets how the array is cut when it is saved: 8 time steps per chunk, like the cloud store. Your selection starts in the middle of a chunk, and a Zarr store needs chunks of equal size.
* `.drop_encoding()` makes `xarray` forget the settings it remembered from the cloud store, such as the compression. Your new store gets its own settings.
* `zarr_format=2` saves the store in Zarr version 2. Its folder layout is the easiest to read. Many newer stores use version 3, which holds the same information in a slightly different layout.

In [ ]:
%%time
store = "./results/t2m_2022.zarr"

(
    t2m_2022
    .chunk({"time": 8})     # 8 time steps per chunk, like the cloud store
    .drop_encoding()        # forget the settings of the cloud store
    .to_dataset()
    .to_zarr(store, zarr_format=2, consolidated=True, mode="w")
)

### 🔍 Step 5: Look inside the folder

A Zarr store is an ordinary folder. Start at the top:

In [ ]:
sorted(os.listdir(store))

The variable and each coordinate have their own sub-folder. The names that start with a dot are metadata files. Next, you look into the sub-folder for temperature.

::: {.callout-tip}
## Reflection and answers

Reading one time-step cost you 1 chunk. What about if you extract the temperature at Snowbird for every time step in 2022:

- ➡️ Before you run the next cell: the array has 1,460 time steps in chunks of 8. How many **files** do you expect in the folder, if there are two metadata files?<br>
    ✏️ : **185 files**. 1,460 ÷ 8 = 182.5, so there are 183 chunk files: 182 full chunks and one half chunk at the end. Two metadata files make 185.
:::

In [ ]:
files = sorted(os.listdir(f"{store}/2m_temperature"))

print(f"{len(files)} files")
print("First files:", files[:4])

There are two kinds of files:

* **Chunk files** are named after their position in the array: `0.0.0` is the first chunk along time, longitude and latitude, `1.0.0` is the next one along time, and so on. Each holds 8 time steps of the whole globe.
* **Metadata files** have names that start with a dot. The most important one is `.zarray`:

In [ ]:
with open(f"{store}/2m_temperature/.zarray") as f:
    print(json.dumps(json.load(f), indent=2))

This small text file is the "contents page" of the array:

<center>

| Line | Meaning |
|---|---|
| **shape** | the size of the whole array: 1,460 time steps × 240 × 121 grid points |
| **chunks** | the size of one chunk: 8 × 240 × 121 |
| **dtype** | the data type of each value, e.g. `<f4` for a 32-bit decimal number |
| **compressor** | how each chunk is compressed before it is saved |
| **fill_value** | the value used for missing data |

</center>

From **shape** and **chunks** alone, `xarray` can work out which chunk file holds any value you ask for, without opening the others. This is what happened when you opened ERA5 in Step 1: `xarray` read files like this one, and nothing else.

::: {.callout-note}
## Note

💡 If you know netCDF...

For `xarray`, a Zarr store and a netCDF file look the same: dimensions, coordinates, variables and attributes. The difference is the packaging. A netCDF file keeps everything in **one file**. A Zarr store keeps each chunk as its **own object**, so a reader in the cloud can fetch just the chunks it needs, many at once.

💡 Remember...

- 📖 **Reading:** opening a store reads only its metadata. Data is downloaded when you load, plot or save, one chunk at a time.
- 💾 **Writing:** `to_zarr()` saves a folder of **metadata files**, which describe each array, and **chunk files**, one per chunk.
- 🧮 **So far:** one map cost you 1 chunk, and one year cost you 183.
- ✍️ Whoever **writes** the store decides the shape of the chunks. In the next section, you see how much that decision matters.
:::

<hr>

<a id="example2"></a>

## Example 2️⃣: About Chunking

In Example 1, the map of the world cost you 1 chunk. Now ask a second question about the same year: **how did the temperature at Snowbird change over 2022?** This time, you need a single grid point at every time step. In this example, you:

1. 🔮 **Predict** what the question costs on your store
2. 🧊 **Write** the same year again and chunk your data in a different way
3. ⏱️ **Ask** both stores both questions, and time them
4. ☁️ **Count** what the question would cost on the cloud store

### 🔮 Step 1: Predict the cost

Your store holds 2022 in 183 chunk files. Each of them holds 8 time steps of the whole globe.

::: {.callout-tip}
## Reflection and answer

- ➡️ 1. What chunk shape would let you extract the time-series for Snowbird with a single chunk?<br>
    ✏️ : A chunk that is **long in time and small in space**: the whole year, but only a small patch of the globe.
:::

### 🧊 Step 2: Write the same year in a second shape

First, open the Zarr Store you save before. It works just like the cloud store, only without `storage_options`. Then save it again, but with different chunking. `.chunk()` sets how the array is split, and `-1` means "the whole dimension":

```text
spatial-shaped:          8 time steps × 240 × 121    → one chunk = the whole globe for 2 days
time-series-shaped:  1,460 time steps × 10 × 10  → one chunk = one small patch for the whole year
```

In [ ]:
t2m_spatial_shaped = xr.open_zarr(store)["2m_temperature"]

series_store = "./results/t2m_2022_series.zarr"

(
    t2m_spatial_shaped
    .chunk({"time": -1, "longitude": 10, "latitude": 10})
    .drop_encoding()
    .to_dataset()
    .to_zarr(series_store, zarr_format=2, consolidated=True, mode="w")
)

t2m_series_shaped = xr.open_zarr(series_store)["2m_temperature"]

In [ ]:
t2m_series_shaped

In [ ]:
print("Spatial-shaped store:        ", t2m_spatial_shaped.encoding["chunks"],
      "in", len(os.listdir(f"{store}/2m_temperature")) - 2, "chunk files")
print("Time-series-shaped store:", t2m_series_shaped.encoding["chunks"],
      "in", len(os.listdir(f"{series_store}/2m_temperature")) - 2, "chunk files")

240 ÷ 10 = 24 chunks along longitude and 13 along latitude, so the new store has 24 × 13 = **312 chunk files**. It is the same data, cut in a completely different way.

### ⏱️ Step 3: Ask both stores both questions

You now have the same year in two shapes. Ask each store the two questions, and note how long each read takes. Run each timing cell twice: the first run can be slower.

Start with the spatial extract. `.load()` forces the read, and `%%time` measures it:

In [ ]:
%%time
spatial_a = t2m_spatial_shaped.sel(time="2022-07-01T12:00").load()

In [ ]:
%%time
spatial_b = t2m_series_shaped.sel(time="2022-07-01T12:00").load()

Now let us extract the time-series for 2022. The longitudes of this dataset run from 0° to 360°, so Snowbird's −111.7° becomes 360 − 111.7 = **248.3°**. `method="nearest"` picks the closest grid point:

In [ ]:
snowbird = dict(latitude=40.6, longitude=248.3, method="nearest")
snowbird

::: {.callout-warning icon=false}
## 📝 Instructions

Read the Snowbird time series from **both** stores and time each read, as you did for the example before.

**Hint:** `.sel(**snowbird)` selects the grid point. You don't need to select a time, because you want every time step.
:::

In [ ]:
%%time
series_a = t2m_spatial_shaped.sel(**snowbird).load()

In [ ]:
%%time
series_b = t2m_series_shaped.sel(**snowbird).load()

The times below are from a test run. Yours will differ, but the pattern is the same:

<center>

| Store | 🗺️ Spatial: chunks | 🗺️ Spatial: time | 📈 Snowbird: chunks | 📈 Snowbird: time |
|---|---|---|---|---|
| Spatial-shaped (183 chunk files) | 1 | ⚡ 7.65 ms | 183 | 🐢 265 ms |
| Time-series-shaped (312 chunk files) | 312 | 🐢 469 ms | 1 | ⚡ 5.68 ms |

</center>

Each store is fast for one question and slow for the other. When the chunks match the question, **one chunk** holds everything you need. When they don't, `xarray` has to read and decompress **every chunk** in the store, only to keep a tiny part of each.

### ☁️ Step 4: Back to the cloud

The cloud store is optimised for spatial analyses. What would the time series extraction cost for all 64 years? You can work it out from the metadata, without downloading anything:

In [ ]:
time_steps = t2m.sizes["time"]
steps_per_chunk = t2m.encoding["chunks"][0]

print("Time steps:    ", time_steps)
print("Chunks to read:", round(time_steps / steps_per_chunk))
print("Data to read:  ", round(t2m.nbytes / 1e9, 1), "GB")

::: {.callout-tip}
## Reflection and answer

- ➡️ 1. Imagine many users need both, spatial and time series analyses. What could a provider do?<br>
    ✏️ : There are two common solutions. A provider can publish **two copies** with different chunk shapes. ECMWF does this for its [ARCO ERA5 Zarr Stores](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=analysis_ready_data), with a "time-chunked" and a "geo-chunked" copy. Or it can choose a **compromise**, e.g. a few weeks × a few hundred km, which is not ideal for either question but acceptable for both.
:::

::: {.callout-note}
## Note

💡 Remember...

There is no single "best" chunk shape. **Chunks follow the question:**

- 🗺️ Questions about **space** (spatial analyses) need chunks that are wide in space and short in time.
- 📈 Questions about **one place over time** (time series) need chunks that are long in time and small in space.

Before you analyse a Zarr dataset, check its chunks with `.encoding["chunks"]`, and ask yourself whether they match your question.
:::

<hr>

<a id="example3"></a>

## Exercise 1️⃣: Explore a Sentinel-2 Zarr product

So far, you were given the address of the store. In practice, you first have to **find** the data, and that is what STAC is for. In *Discover STAC*, you met the **EOPF Sentinel Zarr Samples Service**: the European Space Agency (ESA) is converting the Sentinel archive into Zarr and publishing it in a STAC catalog.

Now you apply what you did with ERA5 to a new store: you inspect its structure, judge its chunks, and read only what you need. Your question: **how green is the Douro valley in late summer?** The valley, in northern Portugal, is famous for its vineyards on steep, terraced slopes.

One thing is new. In ERA5, all variables sat side by side. A Sentinel-2 product holds many kinds of data: reflectance at three resolutions, masks, sun angles and more. They are organised in **groups**, like folders inside the store.

### 🔎 Find a scene and open its store

This part is done for you. The search works as in *Discover COG*: look for clear summer scenes, keep the ones that cover the whole area, and take the one with the fewest clouds.

In [ ]:
eopf_catalog = Client.open("https://stac.core.eopf.eodc.eu/")

douro_bbox = (-7.75, 41.10, -7.45, 41.25)   # west, south, east, north: around Pinhão
summer_2026 = "2026-06-01/2026-09-15"

douro_clear = eopf_catalog.search(
    collections="sentinel-2-l2a",
    datetime=summer_2026,
    bbox=douro_bbox,
    query={"eo:cloud_cover": {"lte": 15}},
).item_collection()

aoi = box(*douro_bbox)
covering = [scene for scene in douro_clear if shape(scene.geometry).contains(aoi)]

# min() returns the scene with the smallest cloud cover
item = min(covering, key=lambda scene: scene.properties["eo:cloud_cover"])

print(len(douro_clear), "clear scenes,", len(covering), "cover the whole area")
print("Chosen scene:", item.id)

Earth Search stores **one COG file per band**. EOPF stores **the whole scene in one Zarr store**, and the `product` asset holds its address. A store with groups opens with `xr.open_datatree()`. As before, only the metadata is read:

In [ ]:
product_url = item.assets["product"].href

dt = xr.open_datatree(product_url, engine="zarr", chunks={})
list(dt.children)

There are three groups at the top:
<center>

| Group | What it holds |
|---|---|
| `measurements` | the measurements themselves, e.g. reflectance |
| `conditions` | information about the conditions, e.g. masks and sun angles |
| `quality` | quality information, e.g. quicklook images |

</center>

You move through the groups like through folders: `dt["measurements"]` opens a group, and `list(dt["measurements"].children)` shows what is inside it.

::: {.callout-warning icon=false}
## 📝 Instructions

- ➡️ 1. **Navigate.** Find the group that holds the reflectance at **20 m** and open it as `r20m`. Which bands does it hold? Are the red band (`b04`) and the near-infrared band (`b8a`) both there?
- ➡️ 2. **Judge the chunks.** Find the 20 m scene classification under `conditions` and open it as `scl`. Into how many chunks is one layer of `b04` divided, and one layer of `scl`? Which of the two lets you read a small area without reading the whole layer?
- ➡️ 3. **Read only what you need.** Clip `b04`, `b8a` and `scl` to the Douro area, keep only vegetation and bare ground, and map the NDVI.

**Hints**

- `list(dt["..."].children)` looks one level down. Keep going until you reach the resolutions.
- `dt["path/to/group"].to_dataset()` turns a group into a Dataset.
- For task 2, look for a group called `mask`, then for the word *classification*. The array inside is called `scl`.
- `.shape` gives the size of a layer, and `.encoding["chunks"]` the size of its chunks.
- Clip as in *Discover COG*: `.sel(x=slice(west, east), y=slice(north, south))`.
- Classes 4 (vegetation) and 5 (not vegetated) are the ones to keep: `keep = scl.isin([4, 5])`, then `.where(keep)`.
- NDVI = (near-infrared − red) / (near-infrared + red).
:::

#### ➡️ 1. Navigate

In [ ]:
print(list(dt["measurements"].children))
print(list(dt["measurements/reflectance"].children))

In [ ]:
r20m = dt["measurements/reflectance/r20m"].to_dataset()
list(r20m.data_vars)

The path is `measurements/reflectance/r20m`. The group holds ten bands, including both `b04` and `b8a`, so you can calculate the NDVI on one grid. In the COG notebook, you had to choose an overview level for each band to make them match.

#### ➡️ 2. Judge the chunks

In [ ]:
print(list(dt["conditions"].children))
print(list(dt["conditions/mask"].children))
print(list(dt["conditions/mask/l2a_classification"].children))

In [ ]:
scl = dt["conditions/mask/l2a_classification/r20m"].to_dataset()["scl"]

print("Size of one layer:", r20m["b04"].shape)
print("b04 chunks:       ", r20m["b04"].encoding["chunks"])
print("scl chunks:       ", scl.encoding["chunks"])

One layer is 5,490 × 5,490 pixels.

- `b04` is cut into chunks of 2,048 × 2,048 pixels: 3 × 3 = **9 chunks**.
- `scl` is stored as **1 chunk** for the whole layer.

For a small area, the chunks of `b04` suit the question: the Douro area touches only 2 of the 9. For `scl`, any area, however small, means reading the whole layer. Chunks follow the question here too, and two arrays in the same store can be cut in different ways.

#### ➡️ 3. Read only what you need



Sentinel-2 uses UTM coordinates in metres. This area lies in UTM zone 29N (`EPSG:32629`), so the bounding box is converted from degrees to metres for you:

In [ ]:
west, south, east, north = transform_bounds("EPSG:4326", "EPSG:32629", *douro_bbox)

In [ ]:
red = r20m["b04"].sel(x=slice(west, east), y=slice(north, south))
nir = r20m["b8a"].sel(x=slice(west, east), y=slice(north, south))
scl = scl.sel(x=slice(west, east), y=slice(north, south))

keep = scl.isin([4, 5])
red = red.where(keep)
nir = nir.where(keep)
print(f"Pixels kept: {float(keep.mean()):.0%}")

ndvi = (nir - red) / (nir + red)

In [ ]:
ndvi.plot(cmap="YlGn", vmin=0, vmax=1, figsize=(9, 5.5), cbar_kwargs={"label": "NDVI"})
plt.title(f"NDVI in the Douro valley, {item.datetime.date()}")
plt.gca().set_facecolor("lightgrey")
plt.gca().set_aspect("equal")
plt.ticklabel_format(style="plain")
plt.xlabel("Easting (m)")
plt.ylabel("Northing (m)")
plt.show()

::: {.callout-tip}
## Reflection and answer
- ➡️ 1. Of everything in the product, which groups and arrays did you actually read?<br>
    ✏️ : Only **three arrays** (`b04`, `b8a` and `scl`) from **two groups**, and of the two bands, only the chunks that cover the Douro area. Everything else in the product was never downloaded.
:::

<hr>

<a id="example4"></a>

## Takeaways

You opened 64 years of global weather in seconds, saved one year of it to your own disk and looked inside, cut that year in two ways to see what the chunk shape costs, and explored a Sentinel-2 product through its groups.

**What you learned**

* 📋 **A Zarr store is a folder.** It holds **metadata files**, which describe each array (shape, chunks, data type, compression), and **chunk files**, one per chunk. Larger stores organise their arrays in **groups**.
* ☁️ **Open from the cloud:** `xr.open_zarr(url, storage_options={"token": "anon"})` opens a public store by reading only its metadata. Data is downloaded when you load, plot or save, one chunk at a time.
* 💾 **Write your own store:** `.sel()` selects by coordinate values, `.chunk()` sets how the array is cut, and `.to_zarr()` saves it.
* 🧊 **Chunks follow the question:** spatial-shaped chunks (wide in space, short in time) are fast for maps. Time-series-shaped chunks (long in time, small in space) are fast for time series. `.encoding["chunks"]` shows how an array is cut.

**Things to keep in mind**

* **Whoever writes the store decides the chunks.** Different datasets, and even different arrays in the same product, can be split very differently. Always check before you analyse.
* When you **save** a Zarr store, all chunks must have the same size, except the last one. Set them with `.chunk()` first.
* As with COGs, check whether values need **scale and offset**. For EOPF, `xarray` applies them for you from the Zarr metadata.
* You looked inside a **version 2** store. Newer stores use **version 3**: the ideas are the same, but the metadata files are called `zarr.json`.

**Next steps**

* COGs and Zarr are formats for gridded (raster) data. The next notebook introduces **GeoParquet**, a cloud-native format for vector data such as points, lines and polygons.
* In *Discover Dask*, you come back to the ERA5 store with a bigger question: how has November at Snowbird changed over 64 years? Many workers then read the chunks at the same time.
* A Zarr store can **grow**: `to_zarr(..., append_dim="time")` adds new time steps to a store that already exists.

📚 *Further reading:* [zarr.dev](https://zarr.dev) · [xarray: Zarr I/O](https://docs.xarray.dev/en/stable/user-guide/io.html#zarr) · [EOPF Sentinel Zarr Samples](https://zarr.eopf.copernicus.eu/) · [WeatherBench 2 data guide](https://weatherbench2.readthedocs.io/en/latest/data-guide.html) · [Cloud-Optimized Geospatial Formats Guide](https://guide.cloudnativegeo.org)

<hr>

&copy; 2026 thriveGEO GmbH
